# AIC25 — Tier-2: 3D-HOTA Numbers

Produces the official **3D-HOTA** score for Warehouse_016 (paper-comparable; Glance-MCMT scored 43–51).

**Same fixes as Tier-1** (OSNet from HF mirror, EmbedFeature local, fresh capped detection) **plus** depth maps + multi-camera + TrackEval.

⚠️ **Heavy path:**
1. **Depth maps** (30–80 GB) — required for 3D world coordinates.
2. **All cameras** — multi-camera HOTA needs every view (`CAMERAS = None`).
3. Single-camera tracking re-run **with depth present**.

`MAXF` caps frames for a faster (partial) HOTA; set `MAXF = 0` for full 9000-frame, paper-faithful HOTA (very long on free Colab).

Branch: **`hithesh/combined-pipeline`**. T4 GPU.

---
## Step 0 — Environment + Drive

In [ ]:
import os, sys, shutil
ON_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
if ON_COLAB:
    REPO='/content/repo'; PY='python'; DRIVE='/content/drive/MyDrive/AIC25'
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'): drive.mount('/content/drive')
    else: print('Drive already mounted.')
    for d in ['models','outputs/Detection','outputs/Tracking']:
        os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
    print('Colab | Drive:', DRIVE)
else:
    REPO='/home/seco/deepLearning/Single-Camera-Tracking-Consistency'; PY=f'{REPO}/.venv/bin/python'; DRIVE=None
    os.chdir(REPO); print('Local')
print('REPO:', REPO)

---
## Step 1 — Clone + checkout combined branch + install

In [ ]:
if ON_COLAB:
    import subprocess as _sp
    if not os.path.exists(REPO):
        os.system(f'git clone https://github.com/Hithesh18/Single-Camera-Tracking-Consistency.git {REPO}')
    else:
        os.system(f'git -C {REPO} fetch --quiet')
    os.chdir(REPO)
    rc = os.system(f'git -C {REPO} checkout hithesh/combined-pipeline')
    os.system(f'git -C {REPO} pull --quiet 2>/dev/null')
    if rc != 0 or not os.path.isdir(f'{REPO}/tracklet_repair'):
        raise RuntimeError('Checkout failed — push: git push -u origin hithesh/combined-pipeline')
    print('On hithesh/combined-pipeline ✓')
    for _p in ['thop','loguru','lap','motmetrics','filterpy','easydict','yacs','termcolor',
               'prettytable','tabulate','ninja','cython_bbox','pycocotools','huggingface_hub','h5py']:
        _sp.run(['pip','install','-q',_p], capture_output=True, text=True)
    if _sp.run(['pip','install','-q','faiss-gpu'], capture_output=True).returncode != 0:
        _sp.run(['pip','install','-q','faiss-cpu'], capture_output=True)
    os.chdir(f'{REPO}/BoT-SORT');         os.system('python setup.py develop --quiet 2>/dev/null')
    os.chdir(f'{REPO}/deep-person-reid'); os.system('python setup.py develop --quiet 2>/dev/null')
    os.chdir(REPO); os.system('pip install -q -r tracking/requirements.txt 2>/dev/null')
    print('Dependencies installed.')
else:
    print('Local: skip.')

---
## Step 2 — GPU check

In [ ]:
import subprocess
r = subprocess.run([PY,'-c','import torch; print("CUDA:", torch.cuda.is_available())'], capture_output=True, text=True)
print(r.stdout.strip())
if ON_COLAB and 'CUDA: False' in r.stdout:
    raise RuntimeError('NO GPU — switch to T4, Restart, re-run.')

---
## Step 3 — Models (OSNet from HF mirror + ByteTrack; AIC25 detector if trained)

In [ ]:
if ON_COLAB:
    from huggingface_hub import hf_hub_download
    M=f'{DRIVE}/models'; os.makedirs(M, exist_ok=True)
    osnet_local=f'{REPO}/deep-person-reid/checkpoints/osnet_ms_m_c.pth.tar'; osnet_drive=f'{M}/osnet_ms_m_c.pth.tar'
    os.makedirs(os.path.dirname(osnet_local), exist_ok=True)
    if os.path.exists(osnet_local): print('OSNet: local')
    elif os.path.exists(osnet_drive): shutil.copy(osnet_drive, osnet_local); print('OSNet: from Drive')
    else:
        fn='osnet_x1_0_msmt17_combineall_256x128_amsgrad_ep150_stp60_lr0.0015_b64_fb10_softmax_labelsmooth_flip_jitter.pth'
        src=hf_hub_download(repo_id='kaiyangzhou/osnet', filename=fn)
        shutil.copy(src, osnet_local); shutil.copy(osnet_local, osnet_drive); print('OSNet: from HF mirror')
    bt_local=f'{REPO}/BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'; bt_drive=f'{M}/bytetrack_x_mot17.pth.tar'
    os.makedirs(os.path.dirname(bt_local), exist_ok=True)
    if not os.path.exists(bt_local):
        if os.path.exists(bt_drive): shutil.copy(bt_drive, bt_local)
        else:
            os.system('pip install -q -U gdown'); import gdown
            gdown.download(id='1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5', output=bt_local, quiet=False)
            if os.path.exists(bt_local): shutil.copy(bt_local, bt_drive)
    aic=f'{M}/ai_city_ckpt.pth.tar'
    if os.path.exists(aic): shutil.copy(aic, f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'); print('AIC25 detector: from Drive ✓')
    else: print('AIC25 detector: not trained — ByteTrack fallback (HOTA will be lower)')
else: print('Local: models in place.')

---
## Step 4 — Config (all cameras for multi-cam HOTA)

In [ ]:
SCENE='Warehouse_016'; DATASET='Val'
CAMERAS=['Camera','Camera_01','Camera_02','Camera_03']  # few cams = fast, fits one session (Subproject-1). Set None for all (needed only for multi-cam HOTA).
MAXF=1500             # frames/camera; 0 = full 9000 (paper-faithful but very long)
TRACK_PARAMS={}       # override BoT-SORT defaults, e.g. {'match_thresh':0.9,'track_buffer':60}; {}=stock (Direction 1 tuning)
os.chdir(REPO)
print(f'{SCENE} ({DATASET}) | cameras ALL | cap {MAXF or "ALL 9000"} frames')

---
## Step 5 — Download videos + ground_truth (HuggingFace)

In [ ]:
if ON_COLAB:
    import getpass
    from huggingface_hub import snapshot_download, login
    dd=f'{DRIVE}/datasets/{DATASET}/{SCENE}'; ld=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}'
    dv=f'{dd}/videos'; vl=f'{ld}/videos'
    os.makedirs(dd, exist_ok=True); os.makedirs(ld, exist_ok=True)
    if os.path.exists(dv) and os.listdir(dv): print('[CACHE HIT] videos on Drive.')
    else:
        tok=None
        try:
            from google.colab import userdata; tok=userdata.get('HF_TOKEN')
        except Exception: pass
        if not tok: tok=getpass.getpass('HF token: ')
        login(token=tok); sp=DATASET.lower()
        print('Downloading videos + GT...', flush=True)
        snapshot_download('nvidia/PhysicalAI-SmartSpaces', repo_type='dataset', local_dir='/content/hf_tmp',
            allow_patterns=[f'MTMC_Tracking_2025/{sp}/{SCENE}/videos/**',
                            f'MTMC_Tracking_2025/{sp}/{SCENE}/calibration.json',
                            f'MTMC_Tracking_2025/{sp}/{SCENE}/ground_truth.json'])
        src=f'/content/hf_tmp/MTMC_Tracking_2025/{sp}/{SCENE}'
        if not os.path.exists(dv): shutil.copytree(f'{src}/videos', dv)
        for fn in ['calibration.json','ground_truth.json']:
            if os.path.exists(f'{src}/{fn}'): shutil.copy(f'{src}/{fn}', f'{dd}/{fn}')
        shutil.rmtree('/content/hf_tmp', ignore_errors=True)
    for fn in ['calibration.json','ground_truth.json']:
        s,d=f'{dd}/{fn}',f'{ld}/{fn}'
        if os.path.exists(s) and not os.path.exists(d): shutil.copy(s,d)
    if os.path.islink(vl): os.unlink(vl)
    os.makedirs(vl, exist_ok=True)
    cams=sorted(os.path.splitext(f)[0] for f in os.listdir(dv) if f.endswith('.mp4'))
    print(f'✓ {len(cams)} cameras: {cams}')
else: print('Local: existing data.')

---
## Step 6 — Download DEPTH MAPS ⚠️ (30–80 GB)
Required for 3D world coords. HF stores them as `depth_map*` ; code expects `depth_map/<Camera>.h5`. Cached to Drive.

In [ ]:
# Depth maps are PER-CAMERA .h5 files and only single_camera_tracking.py reads them.
# We stream them ONE camera at a time in the tracking step below, then delete each,
# so we never need 64 GB at once (free Drive is only 15 GB anyway).
USE_DEPTH = False  # False = skip depth, ground-plane coords (zero download; right for Subproject-1). True = per-camera depth stream for 3D-HOTA.
if ON_COLAB:
    import glob
    from huggingface_hub import snapshot_download
    sp = DATASET.lower()
    local_depth = f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/depth_map'
    if os.path.islink(local_depth): os.unlink(local_depth)
    elif os.path.isdir(local_depth): shutil.rmtree(local_depth, ignore_errors=True)
    os.makedirs(local_depth, exist_ok=True)

    def fetch_camera_depth(cam):
        """Download ONE camera's depth .h5 into local_depth. Returns True if present."""
        dst = f'{local_depth}/{cam}.h5'
        if os.path.exists(dst): return True
        if not USE_DEPTH: return False
        snapshot_download('nvidia/PhysicalAI-SmartSpaces', repo_type='dataset',
            local_dir='/content/hf_depth',
            allow_patterns=[f'MTMC_Tracking_2025/{sp}/{SCENE}/depth_map*/{cam}.h5'])
        hits = glob.glob(f'/content/hf_depth/**/{cam}.h5', recursive=True)
        if hits: shutil.copy(hits[0], dst)
        shutil.rmtree('/content/hf_depth', ignore_errors=True)
        return os.path.exists(dst)

    def drop_camera_depth(cam):
        try: os.remove(f'{local_depth}/{cam}.h5')
        except OSError: pass

    print(f'depth_map/ ready (empty) | USE_DEPTH={USE_DEPTH} | per-camera streaming enabled.')
else:
    def fetch_camera_depth(cam): return True
    def drop_camera_depth(cam): pass
    print('Local: depth_map/ expected under AIC25_Track1/...')


---
## Step 7 — Link outputs (Detection + Tracking → Drive; EmbedFeature → LOCAL)

In [ ]:
if ON_COLAB:
    for folder in ['Detection','Tracking']:
        df=f'{DRIVE}/outputs/{folder}'; rf=f'{REPO}/{folder}'; os.makedirs(df, exist_ok=True)
        if os.path.islink(rf): pass
        elif os.path.isdir(rf):
            for it in os.listdir(rf):
                s,d=f'{rf}/{it}',f'{df}/{it}'
                if not os.path.exists(d): shutil.move(s,d)
            shutil.rmtree(rf, ignore_errors=True); os.symlink(df, rf)
        else: os.symlink(df, rf)
        print(f'{folder}/ → Drive')
    ef=f'{REPO}/EmbedFeature'
    if os.path.islink(ef): os.unlink(ef)
    os.makedirs(ef, exist_ok=True); print('EmbedFeature/ → LOCAL')
else: print('Local.')

---
## Generate single-camera JSONs **with depth** (all cameras, capped)
Depth present → tracking computes 3D world coordinates (needed for HOTA). Fresh capped detection avoids stale-cache hangs.

In [ ]:
import subprocess
os.chdir(REPO)
_dv=f'{DRIVE}/datasets/{DATASET}/{SCENE}/videos' if DRIVE else None
_lv=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/videos'
cam_source=_dv if (_dv and os.path.exists(_dv)) else _lv
all_cams=sorted(os.path.splitext(f)[0] for f in os.listdir(cam_source) if f.endswith('.mp4'))
cams=[c for c in CAMERAS if c in all_cams] if CAMERAS else all_cams
print('Cameras:', cams, '| cap', MAXF or 'ALL')
# --- extract frames first: detection reads videos/<cam>/Frame/*.jpg, NOT the mp4 ---
if ON_COLAB:
    import glob
    vid_dir=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/videos'; os.makedirs(vid_dir, exist_ok=True)
    for cam in cams:
        if os.path.isdir(f'{vid_dir}/{cam}/Frame'): continue
        srcmp4=next((p for p in [f'{cam_source}/{cam}.mp4', f'{cam_source}/{cam}/{cam}.mp4'] if os.path.exists(p)), None)
        if srcmp4 and not os.path.exists(f'{vid_dir}/{cam}.mp4') and not os.path.isdir(f'{vid_dir}/{cam}'):
            os.symlink(srcmp4, f'{vid_dir}/{cam}.mp4')
    os.system(f'{PY} tools/extract_frames_25.py AIC25_Track1/{DATASET} -s {SCENE}')
    for cam in cams:
        n=len(glob.glob(f'{vid_dir}/{cam}/Frame/*.jpg')); print(f'  {cam}: {n} frames')
if os.path.exists(f'{REPO}/BoT-SORT/ai_city_ckpt.pth.tar'):
    ckpt='BoT-SORT/ai_city_ckpt.pth.tar'; exp='BoT-SORT/yolox/exps/example/mot/yolox_x_AI_City_25.py'
else:
    ckpt='BoT-SORT/pretrained/bytetrack_x_mot17.pth.tar'; exp='BoT-SORT/yolox/exps/example/mot/yolox_x_mix_det.py'
det_cap=f'--max_frames {MAXF}' if MAXF else ''; trk_cap=f'--limit_frames {MAXF}' if MAXF else ''
TRACK_ARGS=' '.join(f'--{k} {v}' for k,v in (TRACK_PARAMS or {}).items())
shutil.rmtree(f'{REPO}/Detection/{SCENE}', ignore_errors=True);  os.makedirs(f'{REPO}/Detection/{SCENE}', exist_ok=True)
shutil.rmtree(f'{REPO}/EmbedFeature/{SCENE}', ignore_errors=True)
def run(cmd):
    r=subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if r.returncode!=0: print('\n'.join(r.stdout.strip().splitlines()[-30:]))
    return r.returncode
for cam in cams:
    print(f'\n=== detect {cam} ===')
    run(f'{PY} BoT-SORT/tools/aic25_get_detection.py --scene {SCENE} --dataset {DATASET} --camera {cam} -f {exp} -c {ckpt} {det_cap} ./')
print('\n=== [D] embeddings ===')
emb_rc = run(f'cd {REPO}/deep-person-reid && PYTHONPATH={REPO}/deep-person-reid:{REPO}/deep-person-reid/torchreid {PY} torchreid/aic25_extract.py -s {SCENE} --dataset {DATASET} ../')
os.chdir(REPO)
if emb_rc != 0:
    raise RuntimeError('[D] embeddings FAILED — see traceback above. Tracking needs the .npy files; fix this before continuing.')
for cam in cams:
    print(f'\n=== track+fix {cam} ===')
    has_depth = fetch_camera_depth(cam)          # stream THIS camera's depth .h5 only
    ud = '--use_depth True' if has_depth else ''
    run(f'{PY} BoT-SORT/single_camera_tracking.py -s {SCENE} -c {cam} --dataset {DATASET} {ud} {TRACK_ARGS} {trk_cap}')
    run(f'{PY} BoT-SORT/single_camera_fix.py -s {SCENE} -c {cam} --dataset {DATASET}')
    drop_camera_depth(cam)                        # free disk before next camera
print('\n[DONE] single-camera (with world coords) ready.')

---
## [G] Multi-camera tracking + fix
`multi_camera_revised.py` auto-creates `expN` — we detect it and pass it to `multi_camera_fix.py --exp_path`. `--total_frames` is set to the cap so paths line up.

In [ ]:
import os
os.chdir(REPO)
TF = MAXF if MAXF else 9000
mc_dir=f'{REPO}/Tracking/Multicamera/{SCENE}'
before=set(os.listdir(mc_dir)) if os.path.isdir(mc_dir) else set()
print('[G] multi_camera_revised...')
print('  rc', os.system(f'{PY} BoT-SORT/multi_camera_revised.py -s {SCENE} --dataset {DATASET} --total_frames {TF}'))
after=set(os.listdir(mc_dir)) if os.path.isdir(mc_dir) else set()
EXP=sorted(after-before)[-1] if (after-before) else (sorted(after)[-1] if after else 'exp1')
print('  exp dir =', EXP)
print('[G] multi_camera_fix...')
print('  rc', os.system(f'{PY} BoT-SORT/multi_camera_fix.py -s {SCENE} --dataset {DATASET} --exp_path {EXP}'))
ed=f'{mc_dir}/{EXP}'
print('  exp dir files:', os.listdir(ed) if os.path.isdir(ed) else 'MISSING')
orr=f'{ed}/output_result'
if os.path.isdir(orr): print('  output_result/:', os.listdir(orr))

---
## [H] Evaluate — 3D HOTA
Uses the **same EXP** from [G]. If `fixed_whole_tracking_results.json` isn't found, the [G] file listing shows what `multi_camera_fix` actually produced — adjust if needed.

In [ ]:
import os
os.chdir(f'{REPO}/TrackEval')
print('[H] prepare_eval_data (EXP=%s)...' % EXP)
print('  rc', os.system(f'{PY} prepare_eval_data.py -s {SCENE} --exp {EXP} --dataset {DATASET} --base_dir {REPO}'))
tt=f'{REPO}/TrackEval/aicity_25_data/{SCENE}/{EXP}.txt'
print('  tracker txt exists:', os.path.exists(tt))
print('[H] main (HOTA)...')
print('  rc', os.system(f'{PY} main.py -s {SCENE} --exp {EXP}'))
print('\n--- HOTA / DetA / AssA / LocA printed above ---')

---
## Notes
- **Before/after-repair HOTA** (showing `tracklet_repair` improves the score) needs the repaired single-camera JSON fed back into multi-camera — a schema-adaptation step, not automated here. Do the baseline HOTA first.
- For **paper-faithful** HOTA set `MAXF = 0` (full 9000 frames) — expect a multi-hour run; do it overnight or on a longer session.

---
## Subproject 1 — Single-camera consistency
Ground-truth benchmark · learned ReID matcher · BoT-SORT tuning. All CPU-only except the optional re-tracking sweep. Run after the single-camera step above.

In [ ]:
# [S1-a] deps + ground-truth benchmark: raw vs conservative tracklet_repair, per camera
import os, subprocess, sys
os.chdir(REPO)
subprocess.run([sys.executable,'-m','pip','install','-q','motmetrics'], check=False)
GT=f'{REPO}/AIC25_Track1/{DATASET}/{SCENE}/ground_truth.json'
os.system(f'{PY} -m tracklet_repair.src.evaluation.benchmark_scene '
          f'--gt-json {GT} --scene {SCENE} --merge-mode conservative '
          f'--output-dir tracklet_repair/results/scene_benchmark/{SCENE}_conservative')
print(open(f'{REPO}/tracklet_repair/results/scene_benchmark/{SCENE}_conservative/scene_benchmark.md').read())

In [ ]:
# [S1-b] train the learned ReID matcher on GT-labelled tracklet pairs (CPU)
os.chdir(REPO)
os.system(f'{PY} -m tracklet_repair.src.matcher.train_matcher '
          f'--gt-json {GT} --scene {SCENE} --out tracklet_repair/models/{SCENE}_matcher')

In [ ]:
# [S1-c] benchmark the LEARNED merge vs GT (compare against conservative above)
os.chdir(REPO)
os.system(f'{PY} -m tracklet_repair.src.evaluation.benchmark_scene '
          f'--gt-json {GT} --scene {SCENE} --merge-mode learned '
          f'--matcher-path tracklet_repair/models/{SCENE}_matcher --embed-root EmbedFeature '
          f'--output-dir tracklet_repair/results/scene_benchmark/{SCENE}_learned')
print(open(f'{REPO}/tracklet_repair/results/scene_benchmark/{SCENE}_learned/scene_benchmark.md').read())

### Optional — BoT-SORT parameter tuning (Direction 1)
Re-runs tracking for each parameter combination, so it is slower. Keep the grid and frame cap small. Needs detections + embeddings already present (from the step above).

In [ ]:
# [S1-d] tune BoT-SORT matching thresholds vs GT (re-tracks per combo)
os.chdir(REPO)
_cams=' '.join(cams)
os.system(f'{PY} -m tracklet_repair.src.evaluation.tune_botsort '
          f'--gt-json {GT} --scene {SCENE} --dataset {DATASET} --cameras {_cams} '
          f'--python {PY} --limit-frames {MAXF or 1000} '
          f'--output-dir tracklet_repair/results/botsort_sweep/{SCENE}')
print(open(f'{REPO}/tracklet_repair/results/botsort_sweep/{SCENE}/botsort_sweep.md').read())

In [ ]:
# [S1-e] side-by-side: RAW vs CONSERVATIVE vs LEARNED (key metrics, per camera)
import json
base=f'{REPO}/tracklet_repair/results/scene_benchmark/{SCENE}'
cons=json.load(open(f'{base}_conservative/scene_benchmark.json'))
lrn =json.load(open(f'{base}_learned/scene_benchmark.json'))
def g(d,cam,variant,key): return d['per_camera'][cam]['metrics'][variant][key]
def fmt(v): return f'{v:9.3f}' if isinstance(v,float) and abs(v)<1 else f'{v:9.0f}'
hdr=f"{'camera':12s} {'metric':6s} {'raw':>9s} {'conserv':>9s} {'learned':>9s}"
print(hdr); print('-'*len(hdr))
for cam in cons['per_camera']:
    for key,lbl in [('idf1','IDF1'),('mota','MOTA'),('num_switches','IDSW'),('num_fragmentations','Frag')]:
        raw=g(cons,cam,'baseline',key); c=g(cons,cam,'repaired',key)
        l=g(lrn,cam,'repaired',key) if cam in lrn['per_camera'] else float('nan')
        print(f'{cam:12s} {lbl:6s} {fmt(raw)} {fmt(c)} {fmt(l)}')
    print()
print('Win = learned IDF1 > raw, and learned IDSW/Frag <= raw')